## A Data-Dependent Tile Operation Graph (TOG): Embedding
The compiler writes each kernel's TOG as C++, `tog.cpp`, in the kernel's dump directory, and builds it
into `tog.so`, which TOGSim runs. This notebook compiles an embedding lookup into `tog/embedding/`, so
its TOG can be opened from the file browser.

An embedding reads the table rows its indices name, so the addresses it loads from come from data:
the compiler cannot write them into the TOG. It inserts a record call instead. When Spike runs the
kernel, the call records the addresses (a runtime attribute), and the TOG's load gathers from them.
The machine file turns functional mode on: a timing-only run has no Spike, so it cannot simulate this kernel.

In [ ]:
import os
os.environ["TORCHSIM_TARGET"] = os.path.join(os.getcwd(), "togsim_configs", "togsim_config.yml")
os.environ["TORCHSIM_LOG_PATH"] = os.path.join(os.getcwd(), "togsim_results")
os.environ["TORCHSIM_DUMP_PATH"] = os.path.join(os.getcwd(), "outputs")
import glob
import torch

def latest_log():
    """The newest TOGSim log under TORCHSIM_LOG_PATH."""
    return max(glob.glob(os.path.join(os.environ["TORCHSIM_LOG_PATH"], "*.log")), key=os.path.getmtime)

_seen = set(glob.glob(os.path.join(os.environ["TORCHSIM_LOG_PATH"], "*.log")))

def new_logs():
    """The TOGSim logs written since the last call, oldest first: one per simulation, two for a
    resident mm (its operand's copy-in, then its body), whose cycles add up."""
    logs = set(glob.glob(os.path.join(os.environ["TORCHSIM_LOG_PATH"], "*.log"))) - _seen
    _seen.update(logs)
    return sorted(logs, key=os.path.getmtime)

### Embedding

In [ ]:
os.environ["TORCHSIM_DUMP_PATH"] = os.path.join(os.getcwd(), "tog", "embedding")
device = torch.device("npu:0")
torch.manual_seed(0)

table = torch.randn(8192, 512, dtype=torch.bfloat16).to(device=device)
ids = torch.randperm(8192)[:256].to(device=device)  # 256 distinct rows, in random order

opt_fn = torch.compile(dynamic=False)(torch.nn.functional.embedding)
npu_output = opt_fn(ids, table)
print("matches the CPU:", torch.equal(npu_output.cpu(), table.cpu()[ids.cpu()]))

The kernel's `kernel.json` declares the attribute and the argument it comes from (`in_ptr0`, the indices).

In [ ]:
import json
emb_dir = glob.glob(os.path.join("tog", "embedding", "triton_*"))[0]
print(json.load(open(os.path.join(emb_dir, "kernel.json")))["runtime_attributes"]["attributes"])

What Spike recorded, in `launch-*/runtime/runtime_attribute/attr_<program>_<id>.bin`: past a three-word
header and one extent per dim, one address per element, `512 * id + column`. The 256 lookups run as two programs.


In [ ]:
# the latest launch's files, one per program (attr_<program>_0.bin): the 256 ids split across them
launch = max(glob.glob(os.path.join(emb_dir, "launch-*")), key=os.path.getmtime)
files = sorted(glob.glob(os.path.join(launch, "runtime", "runtime_attribute", "attr_*_0.bin")),
               key=lambda f: int(os.path.basename(f).split("_")[1]))

def recorded(path):
    """One file's index, shaped as it was recorded: magic, version, id and rank as uint32, then the key
    and one extent per dim -- a header of 3 + rank int64 -- and past it one entry per element."""
    raw = bytearray(open(path, "rb").read())
    rank = int(torch.frombuffer(raw, dtype=torch.int32)[3])
    body = torch.frombuffer(raw, dtype=torch.int64)
    return body[3 + rank:].view(body[3:3 + rank].tolist())

# [element of the run, id, chunk of the row]: a row's first element is where that row starts
rows = torch.cat([a[0, :, 0] for a in map(recorded, files)])
print(*files, sep="\n")
print("ids:                ", ids[:6].tolist())
print("recorded row starts:", rows[:6].tolist())
print("row start = 512 * id:", torch.equal(rows, 512 * ids.cpu()))


In the TOG, the table's load gathers from attribute 0. Open `tog.cpp` from the file browser for the
whole graph; `tog_cycles.tsv`, next to it, holds the cycles Gem5 measured for each compute node.

In [ ]:
emb_tog = os.path.join(emb_dir, "tog.cpp")
print(emb_tog)
!grep -n -E "Attribute ATTR|Gather|Desc\(1,|togsim::Load\(" {emb_tog}

### The same TOG, other indices
Each launch records its own addresses, so the cycles follow the data. Both patterns read 256 distinct
rows, so the DRAM reads are the same; each launch's TOGSim log gives them and the bandwidth behind its cycles.

In [ ]:
import re

def launch_stats():
    """Cycles, DRAM reads and DRAM bandwidth (GB/s) of the launches since the last call."""
    cycles, reads, bw = 0, 0, 0.0
    for log in new_logs():
        for l in open(log):
            if "Total execution cycles" in l:
                cycles += int(l.split()[-1])
            m = re.search(r"combined \| ([\d.]+) GB/s aggregate.*\| (\d+) reads", l)
            if m:
                bw, reads = float(m[1]), reads + int(m[2])
    return cycles, reads, bw

launch_stats()
print(f"{'ids':16}{'cycles':>8}{'DRAM reads':>12}{'DRAM GB/s':>11}")
for name, pattern in {"sequential rows": torch.arange(256),
                      "random rows":     ids.cpu()}.items():
    opt_fn(pattern.to(device=device), table)
    cycles, reads, bw = launch_stats()
    print(f"{name:16}{cycles:>8,}{reads:>12,}{bw:>11.0f}")